In [1]:
!pip install networkx -q
!pip install scipy -q

In [2]:
from utils.distances_database_manager import get_distances, get_and_pivot_distances, get_data_with_max_distance
import dask
import dask.dataframe as dd
import pandas as pd
from IPython.display import display, Markdown
import networkx as nx
from tqdm import tqdm
import scipy as sp
from statistics import mean
from dask.diagnostics import ProgressBar
from dask.distributed import Client, progress, LocalCluster
from dask import delayed
import numpy as np

In [3]:
# paths
database_path = '../data_parquet_starpep/'
# database_path = '../db/_temp/'
base_path = '../output/StarPep/PaperNew/2024-12-23T15.40.45-Data_Ingestion/Analysis/'
output_path = './output/graphics/'

In [7]:
# load data
def load_data(distance_function, interval):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[1])]
    
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    
    return data

# repartition data
def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum()
    total_size = total_size.compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data


def construct_graph(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    if group.empty:
        raise ValueError("The provided group is empty. Cannot construct a graph.")
            
    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sequence_length = len(group['sequence'].iloc[0])
    graph.add_nodes_from(range(sequence_length))

    # Agregar las aristas (aminoácidos fuente y destino)
    edges = group[['aminoacid_source', 'aminoacid_target']].values
    graph.add_edges_from(edges)
    
    #sources = group['aminoacid_source'].values
    #targets = group['aminoacid_target'].values
    #graph.add_edges_from(zip(sources, targets))
    
    return graph

def compute_graph_metrics(graph):   
    # Check if the graph is connected and has enough nodes for eigenvector centrality
    if nx.is_connected(graph) and graph.number_of_nodes() > 2:
        try:
            eigenvector_centrality = mean(nx.eigenvector_centrality_numpy(graph).values())
        except Exception:
            eigenvector_centrality = np.nan  # Handle exceptions gracefully
    else:
        eigenvector_centrality = np.nan  # Assign NaN if the graph is disconnected or too small

    # Compute other metrics
    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'eigenvector_centrality': eigenvector_centrality,
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan
    }
    return metrics 

def get_graph_metrics(group, distance_function, interval):
    graph = construct_graph(group, distance_function, interval)
    metrics = compute_graph_metrics(graph)
    return metrics

import os
num_processes = os.cpu_count()

# group_keys=False

filters = [
    {'distance_function': 'euclidean', "interval": (0, 26.2420)},
    {'distance_function': 'euclidean', "interval": (0, 16.6132)},
    {'distance_function': 'euclidean', "interval": (0, 10.2836)},
    
    {'distance_function': 'canberra', "interval": (0, 1.4712)},
    {'distance_function': 'canberra', "interval": (0, 1.0142)},
    {'distance_function': 'canberra', "interval": (0, 0.6155)},
    
    {'distance_function': 'lance_williams', "interval": (0, 0.4660)},
    {'distance_function': 'lance_williams', "interval": (0, 0.3041)},
    {'distance_function': 'lance_williams', "interval": (0, 0.1789)},
    
    {'distance_function': 'clark', "interval": (0, 0.9830)},
    {'distance_function': 'clark', "interval": (0, 0.6813)},
    {'distance_function': 'clark', "interval": (0, 0.4161)},
    
    {'distance_function': 'soergel', "interval": (0, 0.6357)},
    {'distance_function': 'soergel', "interval": (0, 0.4664)},
    {'distance_function': 'soergel', "interval": (0, 0.3035)},
    
    {'distance_function': 'bhattacharyya', "interval": (0, 3.6452)},
    {'distance_function': 'bhattacharyya', "interval": (0, 2.4304)},
    {'distance_function': 'bhattacharyya', "interval": (0, 1.5158)},
    
    {'distance_function': 'angular_separation', "interval": (0, 0.1799)},
    {'distance_function': 'angular_separation', "interval": (0, 0.0654)},
    {'distance_function': 'angular_separation', "interval": (0, 0.0180)}
]


tasks = []
for filter in filters:
    distance_function = filter['distance_function']
    interval = filter['interval']

    # load data
    data = load_data(distance_function, interval) 

    # get graph metric
    metrics = data.groupby('sequence', group_keys=False).apply(
        lambda group: get_graph_metrics(group, distance_function, interval), 
        meta=('graph', 'object')
    )

    tasks.append(metrics)

# Execute in parallel 
dask.config.set(scheduler='processes', num_workers=num_processes) 

#with ProgressBar():
#    computed_metrics = dask.compute(*tasks)

# view result 
#metrics_df = pd.DataFrame(computed_metrics[0].values.tolist())

# Verificar el resultado
#metrics_df.head()

#metrics_df.to_csv("metrics_results.csv", index=False)

#[########################################] | 100% Completed | 21m 17s

# [########################################] | 100% Completed | 31m 34s

In [47]:
len(computed_metrics)

21

In [48]:
metrics_df = pd.DataFrame(computed_metrics[0].values.tolist())
metrics_df.to_csv("metrics_results_1.csv", index=False)

In [52]:
computed_metrics[0].memory_usage(deep=True) / 1e6

17.739512

# variante 2

In [4]:
def load_sequences():
    data = dd.read_parquet(
        database_path,
        columns=['sequence'],
    )
    
    # Obtén secuencias únicas
    unique_sequences = data['sequence'].unique()
    
    # Convierte a DataFrame
    sequences_df = unique_sequences.to_frame(name='sequence')
    
    # Calcula la longitud de cada secuencia
    sequences_df['sequence_length'] = sequences_df['sequence'].str.len()
    
    # Ordena por longitud de secuencia en orden descendente
    sequences_df = sequences_df.sort_values(by='sequence_length', ascending=False)
    
    # Devuelve las secuencias como una lista
    return sequences_df['sequence'].compute().tolist()

s = load_sequences()

In [4]:
def construct_graph(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    if group.empty:
        raise ValueError("The provided group is empty. Cannot construct a graph.")
            
    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sequence_length = len(group['sequence'].iloc[0])
    graph.add_nodes_from(range(sequence_length))

    # Agregar las aristas (aminoácidos fuente y destino)
    edges = group[['aa_src', 'aa_dst']].values
    graph.add_edges_from(edges)
    
    #sources = group['aminoacid_source'].values
    #targets = group['aminoacid_target'].values
    #graph.add_edges_from(zip(sources, targets))
    
    return graph

def compute_graph_metrics(graph):   
    # Check if the graph is connected and has enough nodes for eigenvector centrality
    if nx.is_connected(graph) and graph.number_of_nodes() > 2:
        try:
            eigenvector_centrality = mean(nx.eigenvector_centrality_numpy(graph).values())
        except Exception:
            eigenvector_centrality = np.nan  # Handle exceptions gracefully
    else:
        eigenvector_centrality = np.nan  # Assign NaN if the graph is disconnected or too small

    # Compute other metrics
    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'eigenvector_centrality': eigenvector_centrality,
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan
    }
    return metrics 

def get_graph_metrics(group, distance_function, interval):
    graph = construct_graph(group, distance_function, interval)
    metrics = compute_graph_metrics(graph)
    return metrics
    
filters = [
    {'distance_function': 'euclidean', "interval": (0, 26.2420)},
    {'distance_function': 'canberra', "interval": (0, 1.4712)},
    {'distance_function': 'lance_williams', "interval": (0, 0.4660)},
    {'distance_function': 'clark', "interval": (0, 0.9830)},
    {'distance_function': 'soergel', "interval": (0, 0.6357)},
    {'distance_function': 'bhattacharyya', "interval": (0, 3.6452)},
    {'distance_function': 'angular_separation', "interval": (0, 0.1799)},
    
    {'distance_function': 'euclidean', "interval": (0, 16.6132)},
    {'distance_function': 'canberra', "interval": (0, 1.0142)},
    {'distance_function': 'lance_williams', "interval": (0, 0.3041)},
    {'distance_function': 'clark', "interval": (0, 0.6813)},
    {'distance_function': 'soergel', "interval": (0, 0.4664)},
    {'distance_function': 'bhattacharyya', "interval": (0, 2.4304)},
    {'distance_function': 'angular_separation', "interval": (0, 0.0654)},
    
    {'distance_function': 'euclidean', "interval": (0, 10.2836)},
    {'distance_function': 'canberra', "interval": (0, 0.6155)},
    {'distance_function': 'lance_williams', "interval": (0, 0.1789)},
    {'distance_function': 'clark', "interval": (0, 0.4161)},
    {'distance_function': 'soergel', "interval": (0, 0.3035)},
    {'distance_function': 'bhattacharyya', "interval": (0, 1.5158)},
    {'distance_function': 'angular_separation', "interval": (0, 0.0180)}
]

In [5]:
def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum()
    total_size = total_size.compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data

def load_sequences():
    data = dd.read_parquet(
        database_path,
        columns=['sequence'],
    )

    #sequences = data['sequence'].unique()
    # sequences = sequences.to_frame(name='sequence')    
    # sequences['sequence_length'] = sequences['sequence'].str.len()
    # sequences = sequences.sort_values(by='sequence_length', ascending=False)
    # sequences['sequence'].compute().tolist()
    
    return data['sequence'].unique().compute().tolist()


# Cargar secuencias en lotes de tamaño específico usando yield
def sequence_batches(sequences, batch_size=1000):
    for i in range(0, len(sequences), batch_size):
        yield sequences[i:i + batch_size]


# Modificar la función para que reciba secuencias por lotes
def load_data(distance_function, interval, sequences_batch):
    filters = [
        (distance_function, '>=', interval[0]),
        (distance_function, '<=', interval[1]),
        ('sequence', 'in', sequences_batch)  # Filtrar por el lote de secuencias
    ]
    
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )

    #data['sequence_length'] = data['sequence'].str.len()
    
    # data = data.map_partitions(
    #    lambda df: df.sort_values(by='sequence_length', ascending=False)
    # )

    #data.sort_values(by='sequence_length', ascending=False)    
    #data = data.drop(columns=['sequence_length'])
    
    return data

sequences = load_sequences()

tasks = []
for filter in tqdm(filters, desc='Creating task', total=len(filters)):
    distance_function = filter['distance_function']
    interval = filter['interval']

    for sequences_batch in sequence_batches(sequences, batch_size=1000000000):  # Iterar sobre los lotes de secuencias
        # Cargar los datos para el lote de secuencias
        data = load_data(distance_function, interval, sequences_batch)

        print(data.index.compute())
        
        # Obtener las métricas de los grafos
        metrics = data.groupby('sequence', group_keys=False).apply(
            lambda group: get_graph_metrics(group, distance_function, interval), 
            meta=('graph', 'object')
        )

        tasks.append(metrics)

import os
num_processes = os.cpu_count()

# Ejecutar las tareas en paralelo
dask.config.set(scheduler='processes', num_workers=num_processes) 

with ProgressBar():
    computed_metrics = dask.compute(*tasks)

# batch 1000 secuencias
# [########################################] | 100% Completed | 16m 7ss sin ordenar    
# [########################################] | 100% Completed | 16m 16s  
# [########################################] | 100% Completed | 16m 13s    

# [########################################] | 100% Completed | 18m 7ss ordenando secuencias adentro de cada batch
# [########################################] | 100% Completed | 19m 35s    

# [########################################] | 100% Completed | 20m 39s ordenando solo las secuencias afuera

# [########################################] | 100% Completed | 23m 2ss ordenando afuera y por particion

# batch 10000

Creating task:   5%|▍         | 1/21 [00:02<00:48,  2.42s/it]

Index([       0,        1,        2,        3,        4,        5,        6,
              7,        8,        9,
       ...
       25688936, 25688937, 25688938, 25688939, 25688940, 25688941, 25688942,
       25688943, 25688944, 25688945],
      dtype='int64', length=19266717)


Creating task:  10%|▉         | 2/21 [00:04<00:45,  2.37s/it]

Index([       0,        1,        3,        4,        7,       22,       25,
             26,       27,       28,
       ...
       25688936, 25688937, 25688938, 25688939, 25688940, 25688941, 25688942,
       25688943, 25688944, 25688945],
      dtype='int64', length=19267003)


Creating task:  14%|█▍        | 3/21 [00:07<00:42,  2.34s/it]

...

Index([       1,       25,       26,       27,       28,       49,       50,
             51,       72,       73,
       ...
       25688932, 25688933, 25688934, 25688936, 25688937, 25688938, 25688940,
       25688941, 25688943, 25688945],
      dtype='int64', length=6422477)


Creating task: 100%|██████████| 21/21 [00:50<00:00,  2.40s/it]

Index([      25,      136,      205,      207,      208,      209,      210,
            211,      212,      214,
       ...
       25688936, 25688937, 25688938, 25688939, 25688940, 25688941, 25688942,
       25688943, 25688944, 25688945],
      dtype='int64', length=6416286)


[##########                              ] | 26% Completed | 159.23 s


Process SpawnProcess-1:
Process SpawnProcess-2:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/conda/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/conda/lib/python3.11/concurrent/futures/process.py", line 249, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/opt/conda/lib/python3.11/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
           ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
  File "/opt/conda/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/conda/lib/python3.11/multiprocessing/process.py", line 108, in run
    self.

In [59]:
len(tasks)

735

In [ ]:
# view result 
#metrics_df = pd.DataFrame(computed_metrics[0].values.tolist())

# Verificar el resultado
metrics_df.head()

#metrics_df.to_csv("metrics_results_as.csv", index=False)

In [ ]:

def load_sequences_batch(batch_size=1000):
    # Leer el archivo Parquet en particiones
    data = dd.read_parquet(
        database_path,
        columns=['sequence']
    )

    sequences = data['sequence'].unique().compute().tolist()

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i + batch_size]
        yield batch

In [8]:
import numpy as np
from statistics import mean

def metrics2(graph):
    # Check if the graph is connected and has enough nodes for eigenvector centrality
    if nx.is_connected(graph) and graph.number_of_nodes() > 2:
        try:
            eigenvector_centrality = mean(nx.eigenvector_centrality_numpy(graph).values())
        except Exception:
            eigenvector_centrality = np.nan  # Handle exceptions gracefully
    else:
        eigenvector_centrality = np.nan  # Assign NaN if the graph is disconnected or too small

    # Compute other metrics
    metrics = {
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'eigenvector_centrality': eigenvector_centrality,
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan
    }
    return metrics


import networkx as nx

# Example graph
G = nx.Graph()
G.add_edges_from([(1, 2)])

# Call the metrics2 function
result = metrics2(G)
print(result)


{'number_of_nodes': 2, 'number_of_edges': 1, 'density': 1.0, 'degree_centrality': 1.0, 'eigenvector_centrality': nan, 'closeness_centrality': 1.0, 'betweenness_centrality': 0.0, 'harmonic_centrality': 1.0}


In [32]:
def metrics2(graph):   
    metrics = {
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph).values()) if nx.is_connected(graph) else np.nan,
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values())
    }
    return metrics

In [15]:
G = nx.Graph()
G.add_edges_from([(1, 2), (2, 3)])
nodes = G.nodes()
edges = G.edges()

print(nodes)
print(edges)
print(metrics2(G))

[1, 2, 3]
[(1, 2), (2, 3)]
{'number_of_nodes': 3, 'number_of_edges': 2, 'density': 0.6666666666666666, 'degree_centrality': 0.6666666666666666, 'eigenvector_centrality': 0.5690355937288492, 'closeness_centrality': 0.7777777777777778, 'betweenness_centrality': 0.3333333333333333, 'harmonic_centrality': 1.6666666666666667}


In [16]:
G = nx.Graph()

G.add_nodes_from([5])
G.add_edges_from([(1, 2), (3, 4)])

nodes = G.nodes()
edges = G.edges()

print(nodes)
print(edges)
print(metrics2(G))

[5, 1, 2, 3, 4]
[(1, 2), (3, 4)]
{'number_of_nodes': 5, 'number_of_edges': 2, 'density': 0.2, 'degree_centrality': 0.2, 'eigenvector_centrality': nan, 'closeness_centrality': 0.2, 'betweenness_centrality': 0.0, 'harmonic_centrality': 0.8}


In [13]:
# view result 
metrics_df = pd.DataFrame(computed_metrics[0].values.tolist())

# Verificar el resultado
metrics_df.shape

metrics_df[metrics_df['eigenvector_centrality'].isna()]

,sequence,distance_function,interval,number_of_nodes,number_of_edges,density,degree_centrality,eigenvector_centrality,closeness_centrality,betweenness_centrality,harmonic_centrality


# With batch

In [35]:
# load data
def load_data(sequence, distance_function, interval):
    filters = [('sequence', '==', sequence), (distance_function, '>=', interval[0]), (distance_function, '<=', interval[1])]
    
    data = dd.read_parquet(
        database_path,
        columns=['aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    
    return data

# repartition data
def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum()
    total_size = total_size.compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data


def compute_graph_metrics(graph):   
    # Check if the graph is connected and has enough nodes for eigenvector centrality
    if nx.is_connected(graph) and graph.number_of_nodes() > 2:
        try:
            eigenvector_centrality = mean(nx.eigenvector_centrality_numpy(graph).values())
        except Exception:
            eigenvector_centrality = np.nan  # Handle exceptions gracefully
    else:
        eigenvector_centrality = np.nan  # Assign NaN if the graph is disconnected or too small

    # Compute other metrics
    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'eigenvector_centrality': eigenvector_centrality,
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan,
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values()) if graph.number_of_nodes() > 1 else np.nan
    }
    return metrics 

def empty_graph_metrics(sequence, distance_function, interval):
    metrics = {
        'sequence': sequence,
        'distance_function': distance_function,
        'interval': interval,
        'number_of_nodes': len(sequence),
        'number_of_edges': 0,
        'density': 0,
        'degree_centrality': np.nan,
        'eigenvector_centrality': np.nan,
        'closeness_centrality': np.nan,
        'betweenness_centrality': np.nan,
        'harmonic_centrality': np.nan
    }
    return metrics 

@delayed
def get_graph_metrics(sequence, distance_function, interval):
    # load data
    data = load_data(sequence, distance_function, interval) 

    # get graph
    graph = nx.Graph(to_undirected=True)
    
    graph.graph['sequence'] = sequence
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval    

    metrics = empty_graph_metrics(sequence, distance_function, interval)
    
    if len(data) > 0:      
        graph.add_nodes_from(range(len(sequence)))
        graph.add_edges_from(zip(data['aminoacid_source'].values, data['aminoacid_target'].values))

        # get metrics
        metrics = compute_graph_metrics(graph)
        
    return metrics


def load_sequences():    
    data = dd.read_parquet(
        database_path,
        columns=['sequence'],
    )    
    return data
    
data = load_sequences() 
sequences = data['sequence'].unique().compute().tolist()

import os
num_processes = os.cpu_count()

# group_keys=False

filters = [
   # {'distance_function': 'euclidean', "interval": (0, 26.2420)},
   # {'distance_function': 'euclidean', "interval": (0, 16.6132)},
   #  {'distance_function': 'euclidean', "interval": (0, 10.2836)}
    
   # {'distance_function': 'canberra', "interval": (0, 1.4712)},
   # {'distance_function': 'canberra', "interval": (0, 1.0142)},
   #  {'distance_function': 'canberra', "interval": (0, 0.6155)}
    
   # {'distance_function': 'lance_williams', "interval": (0, 0.4660)},
   # {'distance_function': 'lance_williams', "interval": (0, 0.3041)},
   # {'distance_function': 'lance_williams', "interval": (0, 0.1789)},
    
   # {'distance_function': 'clark', "interval": (0, 0.9830)},
   # {'distance_function': 'clark', "interval": (0, 0.6813)},
   # {'distance_function': 'clark', "interval": (0, 0.4161)},
    
   # {'distance_function': 'soergel', "interval": (0, 0.6357)},
   # {'distance_function': 'soergel', "interval": (0, 0.4664)},
   # {'distance_function': 'soergel', "interval": (0, 0.3035)},
    
   # {'distance_function': 'bhattacharyya', "interval": (0, 3.6452)},
   # {'distance_function': 'bhattacharyya', "interval": (0, 2.4304)},
   # {'distance_function': 'bhattacharyya', "interval": (0, 1.5158)},
    
    #{'distance_function': 'angular_separation', "interval": (0, 0.1799)},
    #{'distance_function': 'angular_separation', "interval": (0, 0.0654)},
    {'distance_function': 'angular_separation', "interval": (0, 0.0180)}
]

tasks = []
for filter in filters:
    distance_function = filter['distance_function']
    interval = filter['interval']

    for sequence in sequences:
        task = get_graph_metrics(sequence, distance_function, interval)    

        tasks.append(task)

# Execute in parallel 
dask.config.set(scheduler='processes', num_workers=num_processes) 

with ProgressBar():
    computed_metrics = dask.compute(*tasks)

# view result 
#metrics_df = pd.DataFrame(computed_metrics[0].values.tolist())

# Verificar el resultado
#metrics_df.head()

#metrics_df.to_csv("metrics_results.csv", index=False)

[                                        ] | 0% Completed | 6.87 s ms


TypeError: Trying to convert <dask_expr.expr.Scalar: expr=ReadParquetFSSpec(020eb3d).size() // 2 > 0, dtype=bool> to a boolean value. Because Dask objects are lazily evaluated, they cannot be converted to a boolean value or used in boolean conditions like if statements. Try calling .compute() to force computation prior to converting to a boolean value or using in a conditional statement.

In [ ]:
dask.config.set(scheduler='processes', num_workers=num_processes) 

with ProgressBar():
    computed_metrics = dask.compute(*tasks)

In [30]:
def load_sequences():
    
    data = dd.read_parquet(
        database_path,
        columns=['sequence'],
    )
    
    return data
    
data = load_sequences() 
sequences = data['sequence'].unique().compute().tolist()

In [32]:
len(sequences[0])

23

In [ ]:
import os
import dask.config
from dask.distributed import Client
import dask.dataframe as dd

# Configuración de Dask para usar múltiples procesos
num_processes = os.cpu_count()
dask.config.set(scheduler='processes', num_workers=num_processes) 

# Función para cargar datos
def load_data(distance_function, interval):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[1])]
    
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    
    return data

# Dividir los grupos de `groupby` en lotes de tamaño 'batch_size'
def batch_groups(group, batch_size):
    # Dividir el grupo en lotes de tamaño 'batch_size'
    num_batches = (len(group) + batch_size - 1) // batch_size  # Calcular el número de lotes
    batches = [group.iloc[i * batch_size:(i + 1) * batch_size] for i in range(num_batches)]  # Crear los lotes
    return batches

# Función para calcular métricas
def compute_graph_metrics(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph).values()),
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values())
    }
    return metrics

# Función para obtener métricas de gráfico
def get_graph_metrics(group, distance_function, interval, batch_size):
    batches = batch_groups(group, batch_size)
    
    metrics_list = []
    for batch in batches:
        metrics = compute_graph_metrics(batch, distance_function, interval)
        metrics_list.append(metrics)
    
    return metrics_list

# batch groups
def batch_groups(groups, batch_size):
    batch = []
    for name, group in groups:
        batch.append((name, group))  # Añadir el grupo completo al batch
        if len(batch) == batch_size:  # Si se alcanza el tamaño del batch
            yield batch
            batch = []  # Reiniciar el batch
    if batch:  # Devolver cualquier grupo restante
        yield batch

filters = [
    #{'distance_function': 'euclidean', "interval": (0, 20)},
    #{'distance_function': 'euclidean', "interval": (0, 15)},
    {'distance_function': 'euclidean', "interval": (0, 10)}
]

tasks = []
batch_size = 100  # Tamaño del lote

# Iterar sobre los filtros
for filter in filters:
    distance_function = filter['distance_function']
    interval = filter['interval']

    # Cargar datos
    data = load_data(distance_function, interval)
    
    # Agrupar por 'sequence' y aplicar el procesamiento por lotes
    groups = data.groupby('sequence')
    
    batches = list(batch_groups(groups, batch_size))
    
    for _, batch in batches:
        get_graph_metrics(batch, distance_function, interval, batch_size)    
   

    tasks.append(metrics)

# Ejecutar las tareas en paralelo con Dask
with ProgressBar():
    computed_metrics = dask.compute(*tasks)


In [ ]:
import os

# Configuración de Dask
num_processes = os.cpu_count()
dask.config.set(scheduler='processes', num_workers=num_processes)

# Cargar datos con Dask
def load_data(distance_function, interval):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[1])]
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    return data

@dask.delayed
def compute_graph_metrics(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)
    
    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph).values()),
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values())
    }
    
    return metrics

# Función para dividir los grupos por secuencia en lotes
def batch_groups(groups, batch_size):
    batch = []
    for name, group in groups:
        batch.append((name, group))  # Añadir el grupo completo al batch
        if len(batch) == batch_size:  # Si se alcanza el tamaño del batch
            yield batch
            batch = []  # Reiniciar el batch
    if batch:  # Devolver cualquier grupo restante
        yield batch

# Crear tareas para cada lote de secuencias
def create_tasks(data, distance_function, interval, batch_size):
    tasks = []
    groups = data.groupby('sequence')
    
    # Crear los lotes de grupos
    for batch in batch_groups(groups, batch_size):
        # Para cada lote, se crea una tarea para calcular las métricas del grafo
        for _, group in batch:
            task = compute_graph_metrics(group, distance_function, interval)
            tasks.append(task)
    
    return tasks

# Configurar filtros y procesar
filters = [{'distance_function': 'euclidean', "interval": (0, 10)}]
batch_size = 100
tasks = []

for filter in filters:
    distance_function = filter['distance_function']
    interval = filter['interval']

    data = load_data(distance_function, interval)

    tasks = create_tasks(data, distance_function, interval, batch_size)
    all_tasks.extend(tasks)

# Ejecutar tareas en paralelo
#with ProgressBar():
#    computed_metrics = dask.compute(*tasks)

# Convertir a DataFrame
#result_df = pd.DataFrame(computed_metrics[0])
#print(result_df.head())


In [17]:
def batch_groups(grouped_data, batch_size):
    # Función que dividirá cada grupo en lotes
    def process_groups(group):
        batch = []
        for name, group_data in group:
            batch.append((name, group_data))
            if len(batch) == batch_size:
                yield batch
                batch = []
        if batch:
            yield batch

    return grouped_data.apply(process_groups, meta=('x', 'object'))

data = load_data('euclidean', (0, 10))
    
# Agrupar los datos por 'sequence'
grouped_data = data.groupby('sequence')

# Crear los lotes aplicando la función personalizada
batches = batch_groups(grouped_data, batch_size=100)



In [ ]:
import os
import dask
import dask.dataframe as dd
from dask import delayed
import networkx as nx
from statistics import mean

# Función para cargar datos
def load_data(distance_function, interval, database_path):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[1])]
    
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    
    return data

# Función para calcular las métricas del gráfico
def compute_graph_metrics(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph).values()),
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values())
    }
    return metrics

# Función para aplicar el cálculo de métricas a cada grupo
def apply_metrics_to_groups(group, distance_function, interval):
    # Aquí aplicamos el cálculo de métricas directamente a cada grupo
    return compute_graph_metrics(group, distance_function, interval)

# Función para crear tareas delayed para cada grupo
def create_metrics_tasks(data, distance_function, interval):
    tasks = []
    
    # Dividir los datos en grupos por 'sequence'
    groups = data.groupby('sequence')

    

    for group in data['sequence'].unique():
        group = groups.get_group(group)
        task = delayed(apply_metrics_to_groups)(group, distance_function, interval)
        tasks.append(task)
    
    return tasks

# Configuración de Dask para usar múltiples procesos
num_processes = os.cpu_count()
dask.config.set(scheduler='processes', num_workers=num_processes) 

filters = [
    {'distance_function': 'euclidean', "interval": (0, 10)}
]

tasks = []
batch_size = 100  # Tamaño del lote

# Iterar sobre los filtros
for filter in filters:
    distance_function = filter['distance_function']
    interval = filter['interval']

    # Cargar datos
    data = load_data(distance_function, interval, database_path)
    
    # Crear tareas delayed para cada grupo
    metrics_tasks = create_metrics_tasks(data, distance_function, interval)
    tasks.extend(metrics_tasks)  # Agregar las tareas a la lista

# Finalmente, ejecutar las tareas en paralelo
#computed_metrics = dask.compute(*tasks)

# Ver el resultado
#print(computed_metrics)


# other

In [39]:
distance_function = 'euclidean'
interval = (0, 10)

data = load_data(distance_function, interval)

group_by_sequence = data.groupby('sequence').apply

In [40]:
data.npartitions

5

In [41]:
data.map_partitions(len).compute()

0    1745121
1    1745122
2     868729
3     868730
4     868730
dtype: int64

In [42]:
data.map_partitions(lambda df: df.memory_usage(deep=True).sum() / 1e6).compute()

0    141.563833
1    125.386033
2     86.384727
3     88.542460
4     64.113010
dtype: float64

In [35]:
data.columns

Index(['sequence', 'aminoacid_source', 'aminoacid_target'], dtype='object')

In [ ]:
%%time
def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum()
    if isinstance(total_size, dd.core.Scalar):  # Verificar si es un objeto Dask
        total_size = total_size.compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data
    
def construct_graph(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    return graph

def compute_graph_metrics(graph):
    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph).values()),
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values())
    }   

    return metrics


def create_group(data, distance_function, interval):
    # filter data
    data_partition = data[(data[distance_function] >= interval[0]) & (data[distance_function] <= interval[1])]
    
    # group data
    grouped = data_partition.groupby('sequence')

    return grouped

@delayed
def process_filter(data, filter):
    distance_function = filter['distance_function']
    interval = filter['interval']

    # filter data
    data_partition = data[(data[distance_function] >= interval[0]) & (data[distance_function] <= interval[1])]
    
    # group data
    grouped = data_partition.groupby('sequence')
    
    results = []
    
    # process group
    for sequence, group in grouped:
        # construct graphs
        graph = construct_graph(group, distance_function, interval)
        
        # compute metrics
        metrics = compute_graph_metrics(graph)
        results.append(metrics)
    
    return results

#config client
#client = Client(processes=True)  # Configurar Dask para usar múltiples procesos
#cluster = LocalCluster()
#client = Client(cluster)

# data
data = dd.read_parquet(database_path)

# partitions
filters = [
    {'distance_function': 'euclidean', "interval": (0, 20)},
    {'distance_function': 'euclidean', "interval": (0, 15)},
    {'distance_function': 'euclidean', "interval": (0, 10)},
    {'distance_function': 'euclidean', "interval": (0, 20)},
    {'distance_function': 'euclidean', "interval": (0, 15)},
    {'distance_function': 'euclidean', "interval": (0, 10)},
    {'distance_function': 'euclidean', "interval": (0, 20)},
    {'distance_function': 'euclidean', "interval": (0, 15)},
    {'distance_function': 'euclidean', "interval": (0, 10)} 
]

# apply filter to data 
#config client
#client = Client(processes=True)

#num_cores = multiprocessing.cpu_count()
#with dask.config.set(pool=ProcessPoolExecutor(num_cores)):
tasks = []
for filter in filters:
    task = process_filter(data, filter)
    tasks.append(task)

# Execute in parallel 
dask.config.set(scheduler='processes') 

with ProgressBar():
    computed_metrics = dask.compute(*tasks)

#client.close()

In [13]:
# Función para cargar datos de Parquet en batches de secuencias
def load_data_in_batches(distance_function, interval, batch_size, database_path):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[1])]
    
    # Cargar el dataframe en Dask, especificando las columnas y filtros
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    
    # Inicializar variables para el batch
    start = 0
    batches = []
    
    # Calcular el total de secuencias
    total_sequences = data['sequence'].nunique().compute()  # Obtén el número total de secuencias
    
    while start < total_sequences:
        # Leer un batch de secuencias, ajustando el índice según `batch_size`
        batch = data.loc[data['sequence'].isin(data['sequence'].unique()[start:start+batch_size])]
        
        batches.append(batch)  # Añadir el batch a la lista
        
        start += batch_size  # Mover el índice al siguiente batch
    
    return batches

# Función para procesar cada batch
def process_batch(batch):
    # Aquí procesas el batch de secuencias
    # Este es solo un ejemplo de cómo se procesan los batches
    return f"Processed batch: {len(batch)} sequences"

# Parámetros
distance_function = 'euclidean'
interval = (0, 10)
batch_size = 1000  # Número de secuencias por batch

# Cargar los datos en batches
batches = load_data_in_batches(distance_function, interval, batch_size, database_path)

tasks = []
for batch in batches:
    task = delayed(process_batch)(batch)
    tasks.append(task)

# Ejecutar las tareas en paralelo
#results = compute(*tasks)

# Mostrar los resultados
#print(results)


NotImplementedError: Passing a 'dask_expr._collection.Series' to `isin`